# Data Engineering
---

## Role & Approach

As the **Data Engineer** on this project, my responsibility is to load, audit, and clean the raw `raw_credit_applications.json` dataset so that it is ready for the downstream **Bias Analysis** (notebook 02) and **Privacy & Governance** assessment (notebook 03).

Every cleaning decision in this notebook is made with two principles in mind:

1. **Fitness for bias analysis.** The Data Scientist needs complete, consistent, and correctly typed records with valid values for the protected attributes `gender` and `date_of_birth`. Records that are structurally invalid or missing these fields cannot contribute to a meaningful fairness audit and must be removed.

2. **No data is permanently lost.** Every record removed from the clean dataset (whether a duplicate, a fraud indicator, or an incomplete application) is preserved in `data/quarantined_records.json` with a machine-readable `_drop_reason` tag. This quarantine file serves as an audit trail and allows a Governance Officer or compliance team to manually review and remediate flagged cases at a later stage.

The six data quality dimensions assessed are: **Uniqueness**, **Consistency**, **Completeness**, **Validity**, **Accuracy**, and **Timeliness**.

## Table of Contents

| # | Section | Quality Dimension |
|---|---------|-------------------|
| 1 | [Data Loading](#1---data-loading) | |
| 2 | [Data Cleaning](#2---data-cleaning) | |
| 2.1 | &nbsp;&nbsp;&nbsp;&nbsp;[Duplicate Records](#1-duplicate-records-uniqueness) | Uniqueness |
| 2.2 | &nbsp;&nbsp;&nbsp;&nbsp;[Mixed Data Types](#2-inconsistent-data-types-consistency) | Consistency |
| 2.3 | &nbsp;&nbsp;&nbsp;&nbsp;[Missing Data](#3-missing-or-incomplete-records-completeness) | Completeness |
| 2.4 | &nbsp;&nbsp;&nbsp;&nbsp;[Categorical Coding](#4-inconsistent-codingformatting-of-categorical-fields-consistency-validity) | Consistency |
| 2.5 | &nbsp;&nbsp;&nbsp;&nbsp;[Impossible Values](#5-invalid-or-impossible-values-accuracy-validity) | Accuracy |
| 2.6 | &nbsp;&nbsp;&nbsp;&nbsp;[Invalid Data Formats](#6-invalid-data-formats-validity) | Validity |
| 3 | [Schema Validation](#3---schema-validation) | |
| 4 | [Export and flatten Data](#4---export-and-flatten-data) | |
| 4.1 | &nbsp;&nbsp;&nbsp;&nbsp;[Quarantine Export](#quarantine-export) | |
| 4.2 | &nbsp;&nbsp;&nbsp;&nbsp;[Cleaned Data Export](#cleaned-data-export) | |

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pymongo', '-q'], check=True)

import pandas as pd
import json
from pymongo import MongoClient
from collections import defaultdict
import re

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1 - Data Loading

In [2]:
# connect to MongoDB and initialize client 
client = MongoClient('mongodb://localhost:27017/')
db = client['novacred_db']
collection = db['applications'] # define collection for more reliability  

# drop collection before loading multiple times
collection.drop()

# load JSON file
with open('../data/raw_credit_applications.json', 'r') as file:
    raw_data = json.load(file)

## 2 - Data Cleaning 

Following the project guidelines, we will address the six core data quality issues:
### 1. Duplicate records (Uniqueness)
MongoDB requires unique `_id` fields. Beyond that, `ssn` (Social Security Number) must also be unique across all records: it is a national identifier that belongs to exactly one individual.

We perform two uniqueness checks:
1. **`_id` duplicates:** same application ID submitted more than once (resubmissions).
2. **SSN duplicates with different names:** two different applicants sharing the same SSN, indicating a data entry error or potential fraud. Neither record can be trusted without manual investigation, so both are removed.

All dropped records are collected in a `quarantined_records` list and exported at the end of the pipeline for human review.

In [3]:
# Initialize quarantine — every record dropped anywhere in the pipeline is collected here
quarantined_records = []

unique_records = {}
duplicates_count = 0

for application in raw_data:
    app_id = application['_id']

    if app_id in unique_records:
        duplicates_count += 1
        print(f"Duplicate found: {app_id}")
        # Quarantine the earlier (overwritten) record — keep the most recent resubmission
        quarantined_records.append({
            **unique_records[app_id],
            "_drop_reason": "duplicate_app_id",
            "_drop_stage": "uniqueness"
        })

    unique_records[app_id] = application

clean_data = list(unique_records.values())

print(f"Found and removed {duplicates_count} duplicate IDs.")
print(f"Quarantined so far: {len(quarantined_records)} record(s).")

Duplicate found: app_042
Duplicate found: app_001
Found and removed 2 duplicate IDs.
Quarantined so far: 2 record(s).


The duplicated documents are resubmitted applications with duplicate IDs. We kept the most recent resubmission.

In [4]:
from collections import defaultdict

# Exclude null/missing SSNs — those are a completeness issue handled later
ssn_map = defaultdict(list)
for app in clean_data:
    ssn = app.get('applicant_info', {}).get('ssn')
    if ssn:
        ssn_map[ssn].append(app)

ssn_conflicts = [(ssn, apps) for ssn, apps in ssn_map.items() if len(apps) > 1]

if not ssn_conflicts:
    print("No SSN conflicts found.")
else:
    conflict_ids = set()
    for ssn, apps in ssn_conflicts:
        for app in apps:
            app['_drop_reason'] = 'ssn_conflict'
            app['_drop_stage']  = 'uniqueness'
            quarantined_records.append(app)
            conflict_ids.add(app['_id'])
        print(f"SSN conflict ({ssn}): dropped {len(apps)} records — {', '.join(a['applicant_info']['full_name'] for a in apps)}")
    clean_data = [app for app in clean_data if app['_id'] not in conflict_ids]

SSN conflict (937-72-8731): dropped 2 records — Sandra Smith, Samuel Hill
SSN conflict (780-24-9300): dropped 2 records — Susan Martinez, Gary Wilson


In [5]:
# Load the fully deduplicated data into MongoDB
collection.insert_many(clean_data)

print(f"Inserted {len(clean_data)} unique documents into MongoDB.")
print(f"Quarantined so far : {len(quarantined_records)} record(s)")

Inserted 496 unique documents into MongoDB.
Quarantined so far : 6 record(s)


### 2. Inconsistent data types (Consistency)
Instead of guessing which fields have inconsistent data types, we will dynamically audit the entire collection. 
This script flattens every document and records the Python data types found for each field. If a field contains more than one data type across the dataset (e.g., both integers and strings), it will be flagged for cleaning.

In [6]:
# Dictionary to hold sets of data types for every field
field_types = defaultdict(set)

# Helper function to flatten nested JSON (e.g., turns {'financials': {'income': 50}} into 'financials.income': 50)
def flatten_doc(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_doc(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # For this audit, we skip complex arrays like spending_behavior
            pass 
        else:
            # We ignore null/None values for the type consistency check
            if v is not None and v != "":
                items.append((new_key, type(v).__name__))
    return dict(items)

# Scan every document in the database
for doc in collection.find({}):
    doc.pop('_id', None) # We don't need to check the MongoDB ID
    flat_doc = flatten_doc(doc)
    
    for field, data_type in flat_doc.items():
        field_types[field].add(data_type)

inconsistent_fields = {field: types for field, types in field_types.items() if len(types) > 1}

if not inconsistent_fields:
    print("No inconsistent data types found.")
else:
    for field, types in inconsistent_fields.items():
        print(f"FLAGGED: '{field}' contains mixed types: {types}")

FLAGGED: 'financials.annual_income' contains mixed types: {'int', 'str', 'float'}


In [7]:
# audit flagged 'financials.annual_income' as having mixed string/int types
# We will dynamically find any strings in these numeric fields and cast them to integers

fields_to_fix = list(inconsistent_fields.keys())

for field in fields_to_fix:
    # Update query: Find documents where this field is a string, and cast it to an int
    result = collection.update_many(
        {field: {"$type": "string"}},
        [{"$set": {field: {"$toInt": f"${field}"}}}]
    )
    if result.modified_count > 0:
        print(f"Fixed: Casted {result.modified_count} string values to integers in '{field}'.")

Fixed: Casted 7 string values to integers in 'financials.annual_income'.


### 3. Missing or incomplete records (Completeness)
Before altering the database, we must audit **all** fields for missing values (nulls, empty strings, or completely missing keys).

In [8]:
# 1. First, we dynamically find EVERY possible field name in our dataset
all_fields = set()
for doc in collection.find({}):
    doc.pop('_id', None)
    all_fields.update(flatten_doc(doc).keys())

# 2. Query MongoDB to count missing values for every single field
missing_report = {}

for field in all_fields:
    missing_count = collection.count_documents({
        "$or": [
            {field: {"$exists": False}},
            {field: None},
            {field: ""}
        ]
    })
    if missing_count > 0:
        missing_report[field] = missing_count

print("--- COMPREHENSIVE MISSING DATA AUDIT ---")
if not missing_report:
    print("No missing data found in any field.")
else:
    for field, count in sorted(missing_report.items()):
        print(f"FLAGGED: '{field}' is missing or empty in {count} records.")

--- COMPREHENSIVE MISSING DATA AUDIT ---
FLAGGED: 'applicant_info.date_of_birth' is missing or empty in 5 records.
FLAGGED: 'applicant_info.email' is missing or empty in 7 records.
FLAGGED: 'applicant_info.gender' is missing or empty in 3 records.
FLAGGED: 'applicant_info.ip_address' is missing or empty in 5 records.
FLAGGED: 'applicant_info.ssn' is missing or empty in 5 records.
FLAGGED: 'applicant_info.zip_code' is missing or empty in 2 records.
FLAGGED: 'decision.approved_amount' is missing or empty in 204 records.
FLAGGED: 'decision.interest_rate' is missing or empty in 204 records.
FLAGGED: 'decision.rejection_reason' is missing or empty in 292 records.
FLAGGED: 'financials.annual_income' is missing or empty in 5 records.
FLAGGED: 'financials.annual_salary' is missing or empty in 491 records.
FLAGGED: 'loan_purpose' is missing or empty in 447 records.
FLAGGED: 'notes' is missing or empty in 494 records.
FLAGGED: 'processing_timestamp' is missing or empty in 434 records.


Based on our comprehensive audit, we have discovered patterns in the missing data. We will apply the following rules:

1. **Drop:** Records missing `ssn`, `date_of_birth`, or `gender` are structurally invalid. Missing `ssn` and `date_of_birth` violate KYC compliance, while missing `gender` compromises our downstream Bias Analysis. These records will be dropped.
2. **Null:** Missing `email` will be set to `null`. Email is non-critical for the credit risk algorithm, so we preserve the rest of the valid financial data.
3. **Impute:** Missing `zip_code` will be filled with `"UNKNOWN_ZIP"`. ZIP code is used for geographic bias analysis so an explicit placeholder is more transparent than null.
4. **Keep as Null:** Conditional fields (like `approved_amount` vs `rejection_reason`) and sparse fields (like `notes`) will be left alone. These will later naturally convert to `NaN` in Pandas.
5. **Rename (Schema Drift):** Five records store income under the legacy key `annual_salary`. It will be renamed to `annual_income` to match the data dictionary.

In [9]:
# Helper: fetch documents matching a query, tag them, and add to quarantine
def quarantine_before_drop(query, reason):
    docs = list(collection.find(query))
    for doc in docs:
        doc['_drop_reason'] = reason
        doc['_drop_stage']  = 'completeness'
        quarantined_records.append(doc)
    return docs

missing_ssn_query    = {"$or": [{"applicant_info.ssn": {"$exists": False}},  {"applicant_info.ssn": None},  {"applicant_info.ssn": ""}]}
missing_dob_query    = {"$or": [{"applicant_info.date_of_birth": {"$exists": False}}, {"applicant_info.date_of_birth": None}, {"applicant_info.date_of_birth": ""}]}
missing_gender_query = {"$or": [{"applicant_info.gender": {"$exists": False}}, {"applicant_info.gender": None}, {"applicant_info.gender": ""}]}

# 1. DROP structurally invalid records — quarantine first, then delete
quarantine_before_drop(missing_ssn_query,    'missing_ssn')
deleted_ssn = collection.delete_many(missing_ssn_query)
print(f"Dropped {deleted_ssn.deleted_count} records (Missing SSN).")

quarantine_before_drop(missing_dob_query,    'missing_date_of_birth')
deleted_dob = collection.delete_many(missing_dob_query)
print(f"Dropped {deleted_dob.deleted_count} records (Missing DOB).")

quarantine_before_drop(missing_gender_query, 'missing_gender')
deleted_gender = collection.delete_many(missing_gender_query)
print(f"Dropped {deleted_gender.deleted_count} records (Missing Gender).")

print(f"Quarantined so far: {len(quarantined_records)} record(s).")

# 2. NULL missing emails
updated_emails = collection.update_many(
    {"$or": [{"applicant_info.email": {"$exists": False}}, {"applicant_info.email": ""}]},
    {"$set": {"applicant_info.email": None}})
print(f"\nNulled {updated_emails.modified_count} missing emails.")

# 3. IMPUTE placeholder for zip_code
updated_zips = collection.update_many(
    {"$or": [{"applicant_info.zip_code": {"$exists": False}}, {"applicant_info.zip_code": None}, {"applicant_info.zip_code": ""}]},
    {"$set": {"applicant_info.zip_code": "UNKNOWN_ZIP"}})
print(f"Imputed {updated_zips.modified_count} missing zip codes.")

# 4. FIX SCHEMA DRIFT (Rename 'annual_salary')
renamed_fields = collection.update_many(
    {"financials.annual_salary": {"$exists": True}},
    {"$rename": {"financials.annual_salary": "financials.annual_income"}})
print(f"Renamed {renamed_fields.modified_count} 'annual_salary' fields to 'annual_income'.")

Dropped 5 records (Missing SSN).
Dropped 1 records (Missing DOB).
Dropped 0 records (Missing Gender).
Quarantined so far: 12 record(s).

Nulled 2 missing emails.
Imputed 0 missing zip codes.
Renamed 5 'annual_salary' fields to 'annual_income'.


Based on the audit results, the 5 records missing an SSN were completely blank "ghost" applications that also accounted for the missing demographic fields (such as gender and zip code). Therefore, dropping these 5 structurally invalid applications simultaneously resolved the missing data issues across those other fields, leaving a clean dataset for the Data Scientist.

### 4. Inconsistent coding/formatting of categorical fields (Consistency (Validity))
To ensure our categorical variables are formatted valid consistently, we first audit the database to extract every unique value present in these fields. Once we identify the inconsistent variations (e.g., abbreviations, varying capitalization), we will apply a strict mapping logic to standardize them.

In [10]:
# Define the known categorical fields
categorical_fields = [
    "applicant_info.gender", 
    "loan_purpose", 
    "decision.rejection_reason"
]

for field in categorical_fields:
    # Use MongoDB's distinct() to grab all unique values currently in the database
    unique_values = collection.distinct(field)
    
    # Filter out None/UNKNOWN just to see the actual messy data
    clean_unique_values = [v for v in unique_values if v not in [None, "UNKNOWN"]]
    
    print(f"\nUnique values found in '{field}':")
    print(clean_unique_values)


Unique values found in 'applicant_info.gender':
['F', 'Female', 'M', 'Male']

Unique values found in 'loan_purpose':
['auto', 'business', 'debt_consolidation', 'education', 'home_improvement', 'medical', 'moving', 'personal', 'vacation', 'wedding']

Unique values found in 'decision.rejection_reason':
['algorithm_risk_score', 'high_dti_ratio', 'insufficient_credit_history', 'low_income']


**Executing the Categorical Standardization:**
Based on the audit results, the `applicant_info.gender` field contains inconsistent coding (mixing "M"/"Male" and "F"/"Female"). We will standardize these to use the full words "Male" and "Female" to ensure accurate demographic groupings for the downstream Bias Analysis.

In [11]:
# We only need to target the specific abbreviations found in our audit
gender_fixes = {
    "M": "Male",
    "F": "Female"
}

total_fixed = 0

for messy_value, standard_value in gender_fixes.items():
    # Find any document with the abbreviated gender and replace it with the full word
    updated_gender = collection.update_many(
        {"applicant_info.gender": messy_value},
        {"$set": {"applicant_info.gender": standard_value}}
    )
    
    if updated_gender.modified_count > 0:
        print(f"Standardized {updated_gender.modified_count} records from '{messy_value}' to '{standard_value}'.")
        total_fixed += updated_gender.modified_count

if total_fixed == 0:
    print("No categorical formatting fixes were needed.")
else:
    print(f"\nFixed a total of {total_fixed} records.")

Standardized 52 records from 'M' to 'Male'.
Standardized 57 records from 'F' to 'Female'.

Fixed a total of 109 records.


### 5. Invalid or impossible values (Accuracy (Validity))
In the context of a credit application, financial figures (like income or debt) and time-based figures (like credit history duration) cannot logically be less than zero. We will dynamically audit the entire database to find any numeric field that contains negative values.

In [12]:
# Helper function to get the actual value from a nested dictionary using a dot-notation string
def get_actual_value(document, dot_notation_key):
    keys = dot_notation_key.split('.')
    current_val = document
    for key in keys:
        if isinstance(current_val, dict):
            current_val = current_val.get(key)
        else:
            return None
    return current_val

# 1. Ensure a list of all fields. 
all_fields = set()
for doc in collection.find({}):
    doc_copy = dict(doc)
    doc_copy.pop('_id', None)
    # Re-using your existing flatten_doc just to grab the keys
    all_fields.update(flatten_doc(doc_copy).keys())

impossible_report = {}

# 2. Dynamically scan all fields for negative numbers and store the documents
for field in all_fields:
    query = {
        "$and": [
            {field: {"$type": ["int", "double", "long", "decimal"]}},
            {field: {"$lt": 0}}
        ]
    }
    
    # Fetch the actual broken documents
    bad_docs = list(collection.find(query))
    
    if len(bad_docs) > 0:
        impossible_report[field] = bad_docs

if not impossible_report:
    print("No negative or impossible numeric values found.")
else:
    for field, docs in sorted(impossible_report.items()):
        print(f"FLAGGED: '{field}' contains {len(docs)} records with negative values.")
        print("Affected documents:")
        
        for doc in docs:
            # Use our new helper function to grab the exact real number!
            bad_value = get_actual_value(doc, field)
            
            print(f"   - Application ID: {doc.get('_id')} | {field} = {bad_value}")

FLAGGED: 'financials.credit_history_months' contains 2 records with negative values.
Affected documents:
   - Application ID: app_043 | financials.credit_history_months = -10
   - Application ID: app_156 | financials.credit_history_months = -3
FLAGGED: 'financials.savings_balance' contains 1 records with negative values.
Affected documents:
   - Application ID: app_290 | financials.savings_balance = -5000


**Quarantining Impossible Values:**
Based on our audit, we identified negative values in both `credit_history_months` and `savings_balance`. Neither can be reliably corrected without manual verification:

- A negative credit history duration could be a sign-entry error, but silently converting it to a positive value would introduce an assumption that may distort the bias analysis.
- A negative savings balance could indicate an overdrawn account or a data entry error. Capping it at `$0` would misrepresent the applicant's actual financial position.

Following the same governance principle applied to missing KYC fields, all 3 affected records are quarantined and removed from the clean dataset. They are tagged with machine-readable `_drop_reason` values and preserved in `data/quarantined_records.json` for manual review.

In [13]:
def quarantine_impossible_values(field, reason):
    """Fetch all records with a negative value in `field`, tag and quarantine them, then delete from DB."""
    query = {
        "$and": [
            {field: {"$type": ["int", "double", "long", "decimal"]}},
            {field: {"$lt": 0}}
        ]
    }
    docs = list(collection.find(query))
    for doc in docs:
        doc['_drop_reason'] = reason
        doc['_drop_stage']  = 'accuracy'
        quarantined_records.append(doc)
    if docs:
        collection.delete_many(query)
        print(f"Quarantined and removed {len(docs)} record(s) with impossible '{field}' values.")
    return len(docs)

n1 = quarantine_impossible_values('financials.credit_history_months', 'impossible_negative_credit_history')
n2 = quarantine_impossible_values('financials.savings_balance',       'impossible_negative_savings_balance')

print(f"\nTotal quarantined in this step: {n1 + n2} record(s).")
print(f"Quarantined so far: {len(quarantined_records)} record(s).")

Quarantined and removed 2 record(s) with impossible 'financials.credit_history_months' values.
Quarantined and removed 1 record(s) with impossible 'financials.savings_balance' values.

Total quarantined in this step: 3 record(s).
Quarantined so far: 15 record(s).


### 6. Invalid data formats (Validity)
Date and time data must follow the ISO 8601 standard (`YYYY-MM-DD` for dates, `YYYY-MM-DDTHH:MM:SS` for timestamps). Contact fields like `email` must also conform to a valid structural format. We audit both fields by flagging any value that deviates from the expected pattern.

In [14]:
# We define the strict ISO patterns we EXPECT to see
date_standards = {
    "applicant_info.date_of_birth": r"^\d{4}-\d{2}-\d{2}$",  # Expected: YYYY-MM-DD
    "processing_timestamp": r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"  # Expected: YYYY-MM-DDTHH:MM:SS
}

inconsistent_dates_found = False

for field, pattern in date_standards.items():
    # Query: Field exists, is a string, but DOES NOT match our strict pattern
    query = {
        field: {"$exists": True, "$type": "string", "$not": {"$regex": pattern}}
    }
    
    total_bad = collection.count_documents(query)
    
    if total_bad > 0:
        inconsistent_dates_found = True
        print(f"\nFLAGGED: '{field}' contains {total_bad} records with non-standard formats.")
        
        # Grab a small sample of the bad documents to see what the mess looks like
        bad_sample = list(collection.find(query).limit(5))
        
        print("Sample of invalid formats found:")
        unique_samples = set()
        for doc in bad_sample:
            # Using our helper function from Step 5 to safely get the nested value
            val = get_actual_value(doc, field)
            unique_samples.add(val)
            
        for sample in unique_samples:
            print(f"   - {sample}")
            
    else:
        print(f"\n '{field}' is perfectly formatted.")

if not inconsistent_dates_found:
    print("\nNo date formatting issues found.")


FLAGGED: 'applicant_info.date_of_birth' contains 156 records with non-standard formats.
Sample of invalid formats found:
   - 1990/07/26
   - 28/01/1990
   - 14/02/1982
   - 18/07/1979
   - 01/12/1978

 'processing_timestamp' is perfectly formatted.


**Executing the Date Format Fix:**
Based on the audit, three distinct slash-delimited formats are present: `DD/MM/YYYY`, `MM/DD/YYYY`, and `YYYY/MM/DD`. Simply replacing slashes with hyphens would not fix the ordering.
We use a **deterministic, rule-based parser** that inspects the numeric magnitude of each segment:

1. **`YYYY/MM/DD`**: first segment > 31, so it must be a 4-digit year. Completely unambiguous.
2. **`DD/MM/YYYY`**: first segment > 12, so it cannot be a month. Unambiguous, European format.
3. **`MM/DD/YYYY`**: second segment > 12, so it must be a day. Unambiguous, American format.
4. **Truly ambiguous**: both first and second segment are ≤ 12 (e.g., `06/07/1990`), so there is no programmatic way to determine intent. We default to European `DD/MM/YYYY` (the majority format in the dataset) and log the count as an acknowledged limitation.

In [15]:
def parse_slash_date(date_str):
    """
    Deterministic rule-based parser for slash-delimited date formats.
    Resolves format by inspecting the numeric magnitude of each segment — no guessing.
    """
    parts = date_str.split('/')
    if len(parts) != 3:
        return None

    a, b, c = parts

    # YYYY/MM/DD: first segment is 4 digits → unambiguous year
    if len(a) == 4:
        return f"{a}-{b.zfill(2)}-{c.zfill(2)}"

    ia, ib = int(a), int(b)

    # DD/MM/YYYY: first segment > 12 → cannot be a month
    if ia > 12:
        return f"{c}-{b.zfill(2)}-{a.zfill(2)}"

    # MM/DD/YYYY: second segment > 12 → cannot be a day in month context
    if ib > 12:
        return f"{c}-{a.zfill(2)}-{b.zfill(2)}"

    # Truly ambiguous (both ≤ 12): default to European DD/MM/YYYY (majority format)
    return f"{c}-{b.zfill(2)}-{a.zfill(2)}"


ambiguous_count = 0
fixed_count = 0

slash_date_docs = list(collection.find({
    "applicant_info.date_of_birth": {"$regex": "/"}
}))

for doc in slash_date_docs:
    raw_date = doc["applicant_info"]["date_of_birth"]
    fixed_date = parse_slash_date(raw_date)

    if fixed_date:
        parts = raw_date.split('/')
        a, b = parts[0], parts[1]
        if len(a) != 4 and int(a) <= 12 and int(b) <= 12:
            ambiguous_count += 1

        collection.update_one(
            {"_id": doc["_id"]},
            {"$set": {"applicant_info.date_of_birth": fixed_date}}
        )
        fixed_count += 1

print(f"Fixed {fixed_count} slash-delimited date(s) to ISO 8601 format.")
print(f"  of which {ambiguous_count} were truly ambiguous → defaulted to European DD/MM/YYYY.")

Fixed 156 slash-delimited date(s) to ISO 8601 format.
  of which 39 were truly ambiguous → defaulted to European DD/MM/YYYY.


Beyond date formats, the `email` field also requires format validation. A value like `"mike johnson@gmail.com"` (space in the address) or `"sarah.smith@"` (no domain) is technically present but not a valid email address. We audit all non-null email values against a standard RFC 5321 pattern (`user@domain.tld`) and set any that fail to `null`.

In [16]:
EMAIL_PATTERN = re.compile(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$')

invalid_email_docs = []
for doc in collection.find({"applicant_info.email": {"$exists": True, "$ne": None}}):
    email_val = doc.get("applicant_info", {}).get("email")
    if email_val is not None and not EMAIL_PATTERN.match(str(email_val)):
        invalid_email_docs.append({"_id": doc["_id"], "email": email_val})

print("--- EMAIL FORMAT AUDIT ---")
print(f"Records with invalid email: {len(invalid_email_docs)}")
for record in invalid_email_docs:
    print(f"  {record['_id']}: {repr(record['email'])}")

--- EMAIL FORMAT AUDIT ---
Records with invalid email: 4
  app_204: 'mike johnson@gmail.com'
  app_299: 'test.user.outlook.com'
  app_068: 'john.doe@invalid'
  app_146: 'sarah.smith@'


In [17]:
invalid_ids = [r["_id"] for r in invalid_email_docs]

if invalid_ids:
    result = collection.update_many(
        {"_id": {"$in": invalid_ids}},
        {"$set": {"applicant_info.email": None}}
    )
    print(f"Set {result.modified_count} invalid/placeholder email(s) to null.")
else:
    print("All emails are valid — no changes needed.")

Set 4 invalid/placeholder email(s) to null.


## 3 - Schema Validation

**Governance Rules Enforced:**
- All applications must contain the `applicant_info` and `financials` objects.
- Critical identifiers (`ssn`) and demographic data (`gender`) must be present and typed as strings.
- `email` must be a valid `user@domain.tld` string or `null` (if no valid address is known).
- `date_of_birth` must conform strictly to ISO 8601 `YYYY-MM-DD` format.
- Financial metrics (`annual_income`, `credit_history_months`, `savings_balance`) must strictly be numeric types.
- Time and savings metrics cannot be logically negative (minimum value of 0).

In [18]:
# 1. Define the strict rules based on the official Data Dictionary
validation_schema = {
    "bsonType": "object",
    "required": ["applicant_info", "financials", "decision"],
    "properties": {
        "applicant_info": {
            "bsonType": "object",
            "required": ["full_name", "email", "ssn", "ip_address", "gender", "date_of_birth", "zip_code"],
            "properties": {
                "full_name": {"bsonType": "string"},
                "email": {
                    # Null when no valid address is known (missing or malformed).
                    # When a string is present, it must match standard RFC 5321 format.
                    "bsonType": ["string", "null"],
                    "pattern": r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$",
                    "description": "Valid email (user@domain.tld) or null if unknown/unverifiable"
                },
                "ssn": {"bsonType": "string"},
                "ip_address": {"bsonType": "string"},
                "gender": {"bsonType": "string"},
                "date_of_birth": {
                    "bsonType": "string",
                    "pattern": r"^\d{4}-\d{2}-\d{2}$",
                    "description": "Must be a string in YYYY-MM-DD format"
                },
                "zip_code": {"bsonType": "string"}
            }
        },
        "financials": {
            "bsonType": "object",
            "required": ["annual_income", "credit_history_months", "debt_to_income", "savings_balance"],
            "properties": {
                "annual_income": {"bsonType": ["int", "double", "long", "decimal"]},
                "credit_history_months": {
                    "bsonType": ["int", "long"],
                    "minimum": 0
                },
                "debt_to_income": {"bsonType": ["int", "double", "long", "decimal"]},
                "savings_balance": {
                    "bsonType": ["int", "double", "long", "decimal"],
                    "minimum": 0
                }
            }
        },
        "spending_behavior": {
            "bsonType": ["array", "null"],
            "items": {
                "bsonType": "object",
                "required": ["category", "amount"],
                "properties": {
                    "category": {"bsonType": "string"},
                    "amount": {"bsonType": ["int", "double", "long", "decimal"]}
                }
            }
        },
        "decision": {
            "bsonType": "object",
            "required": ["loan_approved"],
            "properties": {
                "loan_approved": {"bsonType": "bool"},
                "interest_rate": {"bsonType": ["int", "double", "long", "decimal"]},
                "approved_amount": {"bsonType": ["int", "double", "long", "decimal"]},
                "rejection_reason": {"bsonType": "string"}
            }
        }
    }
}

# 2. Lock the database for the future
db.command({
    "collMod": "applications",
    "validator": {"$jsonSchema": validation_schema},
    "validationLevel": "strict",
    "validationAction": "error"
})
print("Database locked: Comprehensive data dictionary schema applied.")

# 3. THE ULTIMATE TEST: Check the existing data against the rules
invalid_docs_count = collection.count_documents({
    "$nor": [{"$jsonSchema": validation_schema}]
})

if invalid_docs_count == 0:
    print("0 invalid documents found.")
else:
    print(f"ERROR: Found {invalid_docs_count} documents that still violate the schema.")
    bad_docs = collection.find({"$nor": [{"$jsonSchema": validation_schema}]}).limit(5)
    print("Sample of failing documents:")
    for doc in bad_docs:
        print(f"   -> Application ID: {doc.get('_id')}")

Database locked: Comprehensive data dictionary schema applied.
0 invalid documents found.


## 4 - Export and flatten Data

### Quarantine Export

All records removed during the pipeline are exported to `data/quarantined_records.json`. Each record retains its original fields and is tagged with:
- `_drop_reason`: why it was removed (e.g. `missing_ssn`, `ssn_conflict`, `impossible_negative_credit_history`, `impossible_negative_savings_balance`)
- `_drop_stage`: which pipeline stage removed it (`uniqueness`, `completeness`, or `accuracy`)

This file is **not** used in any downstream analysis. Its purpose is purely for audit and manual remediation. A compliance officer can inspect the SSN conflict records to determine whether fraud occurred and take appropriate action.

In [19]:
# Export quarantined records — convert ObjectId to string for JSON serialization
for record in quarantined_records:
    if '_id' in record:
        record['_id'] = str(record['_id'])

with open('../data/quarantined_records.json', 'w') as f:
    json.dump(quarantined_records, f, indent=4, default=str)

print(f"Exported {len(quarantined_records)} quarantined record(s) to data/quarantined_records.json")

Exported 15 quarantined record(s) to data/quarantined_records.json


### Cleaned Data Export

With the database locked and validated, the final step in the Data Engineering pipeline is to extract the clean data. We will export the data into two formats:
1. **JSON:** A clean, nested JSON file for application developers and NoSQL backups.
2. **CSV:** A flattened, tabular CSV file for the Data Science team.

In [20]:
# 1. Fetch all clean records from MongoDB into a list
data_list = list(collection.find({}))

# 2. Export to JSON (Preserves the nested structure)
with open('../data/clean_credit_applications.json', 'w') as json_file:
    json.dump(data_list, json_file, indent=4)

# 3. Use pd.json_normalize to flatten the nested JSON (e.g., applicant_info.gender)
clean_df = pd.json_normalize(data_list)

# 4. Export to CSV (Flattened tabular structure)
clean_df.to_csv('../data/clean_credit_applications.csv', index=False)

print("Exported nested clean data to JSON: data/clean_credit_applications.json")
print("Exported flattened clean data to CSV: data/clean_credit_applications.csv")
print(f"Total clean records ready for analysis: {len(clean_df)}")

Exported nested clean data to JSON: data/clean_credit_applications.json
Exported flattened clean data to CSV: data/clean_credit_applications.csv
Total clean records ready for analysis: 487
